In [5]:
# ==========================================
# WAREHOUSE ANALYTICS PROJECT
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# LOAD DATA
# ------------------------------------------

df = pd.read_csv("warehouse_transactions.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows")
print(df.head())

# ------------------------------------------
# DATA CLEANING
# ------------------------------------------

df["date"] = pd.to_datetime(df["date"])

# ------------------------------------------
# KPI 1 - INVENTORY ACCURACY %
# ------------------------------------------

inventory_accuracy = (
    df["physical_qty"].sum()
    / df["system_qty"].sum()
) * 100

print("\nInventory Accuracy %")
print(round(inventory_accuracy,2))

# ------------------------------------------
# KPI 2 - WAREHOUSE UTILIZATION %
# ------------------------------------------

df["utilization_pct"] = (
    df["used_area_sqm"]
    / df["storage_area_sqm"]
) * 100

avg_utilization = df["utilization_pct"].mean()

print("\nAverage Warehouse Utilization %")
print(round(avg_utilization,2))

# ------------------------------------------
# KPI 3 - WAREHOUSE WISE UTILIZATION
# ------------------------------------------

warehouse_utilization = (
    df.groupby("warehouse")["utilization_pct"]
      .mean()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nWarehouse Utilization")
print(warehouse_utilization)

# ------------------------------------------
# KPI 4 - PICKER PRODUCTIVITY
# ------------------------------------------

picker_productivity = (
    df.groupby("picker")
      .agg(
          total_pick_qty=("pick_qty","sum"),
          total_pick_time=("pick_time_minutes","sum")
      )
)

picker_productivity["units_per_minute"] = (
    picker_productivity["total_pick_qty"]
    / picker_productivity["total_pick_time"]
)

picker_productivity = (
    picker_productivity
    .sort_values(
        "units_per_minute",
        ascending=False
    )
)

print("\nPicker Productivity")
print(picker_productivity)

# ------------------------------------------
# KPI 5 - TOP MOVING SKUS
# ------------------------------------------

top_skus = (
    df.groupby("sku")["pick_qty"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nTop 10 Moving SKUs")
print(top_skus.head(10))

# ------------------------------------------
# KPI 6 - WAREHOUSE PICK VOLUME
# ------------------------------------------

warehouse_pick_volume = (
    df.groupby("warehouse")["pick_qty"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nWarehouse Pick Volume")
print(warehouse_pick_volume)

# ------------------------------------------
# KPI 7 - DAILY TRANSACTIONS
# ------------------------------------------

daily_transactions = (
    df.groupby("date")
      .size()
      .reset_index(name="transactions")
)

print("\nDaily Transactions")
print(daily_transactions.head())

# ------------------------------------------
# KPI 8 - MONTHLY PICK QTY
# ------------------------------------------

monthly_pick_qty = (
    df.groupby(
        df["date"].dt.to_period("M")
    )["pick_qty"]
    .sum()
    .reset_index()
)

print("\nMonthly Pick Quantity")
print(monthly_pick_qty)

# ------------------------------------------
# KPI 9 - INVENTORY VARIANCE
# ------------------------------------------

df["inventory_variance"] = (
    df["physical_qty"]
    - df["system_qty"]
)

inventory_variance = (
    df.groupby("warehouse")["inventory_variance"]
      .sum()
      .reset_index()
)

print("\nInventory Variance")
print(inventory_variance)

# ------------------------------------------
# KPI 10 - TOP PICKERS
# ------------------------------------------

top_pickers = (
    df.groupby("picker")["pick_qty"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nTop Pickers")
print(top_pickers)

# ------------------------------------------
# EXPORT RESULTS
# ------------------------------------------

warehouse_utilization.to_csv(
    "warehouse_utilization.csv",
    index=False
)

picker_productivity.to_csv(
    "picker_productivity.csv"
)

top_skus.to_csv(
    "top_moving_skus.csv",
    index=False
)

warehouse_pick_volume.to_csv(
    "warehouse_pick_volume.csv",
    index=False
)

inventory_variance.to_csv(
    "inventory_variance.csv",
    index=False
)

print("\nAnalysis Complete!")
print("CSV files exported successfully.")

Shape: (5000, 11)

Columns:
['transaction_id', 'date', 'warehouse', 'sku', 'system_qty', 'physical_qty', 'picker', 'pick_qty', 'pick_time_minutes', 'storage_area_sqm', 'used_area_sqm']

First 5 Rows
  transaction_id        date warehouse      sku  system_qty  physical_qty  \
0       TXN00001  2025-04-25  North_DC  SKU0072         101            98   
1       TXN00002  2025-02-17  North_DC  SKU0112        1259          1244   
2       TXN00003  2025-08-03  North_DC  SKU0113        1199          1167   
3       TXN00004  2025-06-24   East_DC  SKU0143        1707          1674   
4       TXN00005  2025-07-14  South_DC  SKU0050        1613          1561   

      picker  pick_qty  pick_time_minutes  storage_area_sqm  used_area_sqm  
0  Picker_04       190                 19              9012           7615  
1  Picker_08       109                  4              5488           4444  
2  Picker_15        51                 43             13928          10663  
3  Picker_05         2        